# 0. Download MIND raw releases and create local splits

**Recovery/from-scratch route.** Most teammates should start at step 1 instead: it reuses local splits or
fetches the team's agreed splits from Drive. This notebook is independent of that shared Drive folder.

It downloads the train and labelled dev releases linked by the [MIND website](https://msnews.github.io/),
extracts them into local `train/` and `test/`, then creates the chronological 85/15 train/validation split.
The official dev release becomes this project's test set; the unlabelled official test release is not used.

Existing complete local splits are preserved unless `REBUILD_SPLITS=True`. A from-scratch stable sort can differ
from the saved team assignment within the boundary second; regenerate dependent datasets after rebuilding.

The website's published links returned HTTP 401 during implementation (17 October 2026); the old Azure URLs
returned HTTP 409. If source access is unavailable, edit the URLs to current working website links or place
`MINDlarge_train.zip` and `MINDlarge_dev.zip` in `0_Data_Acquisition/downloads/` before running. Original extracted
raw files can also be used. This notebook reports access failures; it cannot bypass source permissions.

In [ ]:
import json
import os
import shutil
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import polars as pl
import pyarrow.parquet as pq
import requests
from IPython.display import display

DATA_ROOT = None
if DATA_ROOT is not None:
    ROOT = Path(DATA_ROOT).expanduser().resolve()
elif os.environ.get("NEWSREC_ROOT"):
    ROOT = Path(os.environ["NEWSREC_ROOT"]).expanduser().resolve()
else:
    here = Path.cwd().resolve()
    ROOT = next(
        (p for p in (here, *here.parents) if (p / "run_all.py").is_file()),
        here.parent if here.name == "0_Data_Acquisition" else here,
    )
# These are the links currently published on https://msnews.github.io/.
# If they change, edit the URLs here; existing local ZIPs/raw files can also be used.
TRAIN_URL = (
    "https://huggingface.co/datasets/yjw1029/MIND/resolve/main/MINDlarge_train.zip"
)
DEV_URL = "https://huggingface.co/datasets/yjw1029/MIND/resolve/main/MINDlarge_dev.zip"
REBUILD_SPLITS = os.environ.get("NEWSREC_REBUILD_SPLITS") == "1"
ARCHIVE_DIR = ROOT / "0_Data_Acquisition/downloads"
SPLIT_MANIFEST = ROOT / "0_Data_Acquisition/split_manifest.json"
SPLITS = ("train", "validation", "test")
RELEASE_FILES = (
    "behaviors.tsv",
    "news.tsv",
    "entity_embedding.vec",
    "relation_embedding.vec",
)
print("Local data root:", ROOT)

## Download, extract and split helpers

All implementation is inline. Downloads use resumable staging files; ZIP extraction writes only the known release filenames. A build marker prevents step 1 from consuming an interrupted mixed split.

In [ ]:
def usable_file(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return False
    try:
        if path.suffix == ".parquet":
            pq.read_metadata(path)
        else:
            with path.open("rb") as stream:
                first = stream.readline(65536).lower()
            if b"<html" in first or b"<!doctype" in first:
                return False
        return True
    except (OSError, ValueError):
        return False


def valid_release_zip(path):
    try:
        with zipfile.ZipFile(path) as archive:
            names = [
                Path(info.filename).name
                for info in archive.infolist()
                if not info.is_dir()
            ]
            return all(names.count(name) == 1 for name in RELEASE_FILES)
    except (OSError, zipfile.BadZipFile):
        return False


def download_release_zip(url, destination):
    destination = Path(destination)
    if destination.exists():
        if valid_release_zip(destination):
            print("Using saved archive:", destination.name)
            return destination
        raise ValueError(
            f"Invalid saved ZIP: {destination}. Rename/remove it before retrying."
        )
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".part")
    offset = partial.stat().st_size if partial.exists() else 0
    headers = {"Range": f"bytes={offset}-"} if offset else {}
    if urlparse(url).hostname == "huggingface.co" and os.environ.get("HF_TOKEN"):
        headers["Authorization"] = "Bearer " + os.environ["HF_TOKEN"]
    try:
        with requests.get(
            url, headers=headers, stream=True, timeout=(20, 60)
        ) as response:
            if response.status_code == 416 and valid_release_zip(partial):
                partial.replace(destination)
                return destination
            if response.status_code not in (200, 206):
                raise RuntimeError(
                    f"{destination.name}: source returned HTTP {response.status_code}. Check the current links at https://msnews.github.io/ and edit TRAIN_URL/DEV_URL, or place a downloaded {destination.name} in {destination.parent}. This recovery route does not use the shared Google Drive folder."
                )
            append = offset > 0 and response.status_code == 206
            if response.status_code == 206:
                actual = response.headers.get("Content-Range", "")
                expected_offset = offset if append else 0
                if not actual.startswith(f"bytes {expected_offset}-"):
                    raise RuntimeError(
                        "Server returned an unexpected byte range; partial file was kept."
                    )
            mode = "ab" if append else "wb"
            received = 0
            content_length = response.headers.get("Content-Length")
            print("Downloading:", destination.name, flush=True)
            with partial.open(mode) as output:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        output.write(chunk)
                        received += len(chunk)
            if content_length is not None and received != int(content_length):
                raise RuntimeError(
                    "Incomplete transfer; rerun to resume the partial archive."
                )
        if not valid_release_zip(partial):
            raise RuntimeError(
                "Downloaded archive is incomplete or lacks the four MIND release files."
            )
        partial.replace(destination)
        return destination
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Could not download {destination.name}. Partial bytes are kept for retry; a valid local ZIP/raw release can be used instead."
        ) from exc


def extract_release(archive_path, folder, needed):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for name in needed:
            target = folder / name
            if usable_file(target):
                continue
            members = [
                i
                for i in archive.infolist()
                if not i.is_dir() and Path(i.filename).name == name
            ]
            if len(members) != 1:
                raise ValueError(f"Archive must contain exactly one {name}")
            staging = target.with_name(target.name + ".extracting")
            with archive.open(members[0]) as source, staging.open("wb") as output:
                shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
            if staging.stat().st_size == 0:
                raise ValueError(f"Empty archive member: {name}")
            staging.replace(target)
            print("Extracted:", target)


def read_behaviors(path):
    return pl.read_csv(
        path,
        separator="\t",
        has_header=False,
        quote_char=None,
        new_columns=["session_id", "user_id", "timestamp", "history", "impressions"],
    )


def read_news(path):
    return pl.read_csv(
        path,
        separator="\t",
        has_header=False,
        quote_char=None,
        new_columns=[
            "news_id",
            "category",
            "subcategory",
            "title",
            "abstract",
            "url",
            "title_entities",
            "abstract_entities",
        ],
    )


def referenced_news(behaviors, news):
    history_ids = (
        behaviors.select(
            pl.col("history").drop_nulls().str.split(" ").explode(empty_as_null=True)
        )
        .filter(pl.col("history") != "")
        .rename({"history": "news_id"})
    )
    slate_ids = behaviors.select(
        pl.col("impressions").str.split(" ").explode(empty_as_null=True)
    ).select(pl.col("impressions").str.extract("^(N\\d+)-[01]$", 1).alias("news_id"))
    ids = pl.concat([history_ids, slate_ids]).drop_nulls().unique()["news_id"]
    return news.filter(pl.col("news_id").is_in(ids.implode()))


def write_split_parquet(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    staging = path.with_name(path.name + ".building")
    frame.write_parquet(staging)
    pq.read_metadata(staging)
    staging.replace(path)


def prepare_from_mind(root, train_url, dev_url, rebuild=False):
    root = Path(root)
    split_paths = [
        root / split / f"{kind}_{split}.parquet"
        for split in SPLITS
        for kind in ("behaviors", "news")
    ]
    marker = root / "0_Data_Acquisition/split_manifest.json"
    try:
        previous = json.loads(marker.read_text()) if marker.exists() else {}
    except ValueError:
        previous = {"status": "building"}
    need_split = (
        rebuild
        or not all(usable_file(p) for p in split_paths)
        or previous.get("status") == "building"
    )
    for split, filename, url in (
        ("train", "MINDlarge_train.zip", train_url),
        ("test", "MINDlarge_dev.zip", dev_url),
    ):
        needed = RELEASE_FILES if need_split else ("entity_embedding.vec",)
        if all(usable_file(root / split / name) for name in needed):
            print("Local release files ready:", split)
            continue
        archive = download_release_zip(
            url, root / "0_Data_Acquisition/downloads" / filename
        )
        extract_release(archive, root / split, needed)
    if not need_split:
        print("The complete local splits are preserved. No splitting needed.")
        return {"status": "reused", "root": str(root)}
    marker.parent.mkdir(parents=True, exist_ok=True)
    metadata = {
        "status": "building",
        "source": "MIND website/raw releases",
        "train_fraction": 0.85,
        "tie_policy": "stable source order at equal timestamps",
        "official_dev_used_as": "project test",
    }
    marker.write_text(json.dumps(metadata, indent=2) + "\n")
    behaviors = (
        read_behaviors(root / "train/behaviors.tsv")
        .with_columns(
            pl.col("timestamp").str.to_datetime(format="%m/%d/%Y %I:%M:%S %p")
        )
        .sort("timestamp", maintain_order=True)
    )
    cut = int(behaviors.height * 0.85)
    train, validation = (behaviors.slice(0, cut), behaviors.slice(cut))
    news = read_news(root / "train/news.tsv")
    for split, frame in (("train", train), ("validation", validation)):
        write_split_parquet(frame, root / split / f"behaviors_{split}.parquet")
        selected = referenced_news(frame, news)
        write_split_parquet(selected, root / split / f"news_{split}.parquet")
    train_rows, validation_rows = (train.height, validation.height)
    del behaviors, train, validation, news, selected, frame
    test = read_behaviors(root / "test/behaviors.tsv").with_columns(
        pl.col("timestamp").str.to_datetime(format="%m/%d/%Y %I:%M:%S %p")
    )
    write_split_parquet(test, root / "test/behaviors_test.parquet")
    write_split_parquet(
        read_news(root / "test/news.tsv"), root / "test/news_test.parquet"
    )
    metadata.update(
        status="complete",
        train_rows=train_rows,
        validation_rows=validation_rows,
        test_rows=test.height,
    )
    marker.write_text(json.dumps(metadata, indent=2) + "\n")
    return metadata

In [ ]:
summary = prepare_from_mind(ROOT, TRAIN_URL, DEV_URL, REBUILD_SPLITS)
display(summary)

Continue with `1_Data_Preparation/clean_and_validate_data.ipynb`. The generated inputs are on disk in `train/`, `validation/`, and `test/`; no upload to Drive is needed.